#### The framework:

1. Create the embedding: we use an embedding method(an open source model) to convert query and documents to vectors.

2. Metric measurement: we use a similarity metric(Cosine similarity, euclidean distance or dot product) to determine how close each document is to the query

3. Sorting: we sort the documents by their similarity score and select the top few as the most relevant

- Cosine similarity or Dot product: a higher score indicates greater similarity between two vectors

- Euclidean distance: lower score indicates greater similarity

In [2]:
import numpy as np
import os

In [3]:
def cosine_similarity(v1, array_of_vectors):
    v1 = np.array(v1)
    similarities = []
    
    if len(np.shape(array_of_vectors)) == 1:
        array_of_vectors = [array_of_vectors]
        
    for v2 in array_of_vectors:
        v2 = np.array(v2)
        
        dot_product = np.dot(v1, v2)
        norm_v1 = np.linalg.norm(v1)
        norm_v2 = np.linalg.norm(v2)
        
        similarity = dot_product/ (norm_v1 * norm_v2)
        similarities.append(similarity)
    return [float(x) for x in similarities]

In [4]:
def euclidean_distance(v1, array_of_vectors):
    v1 = np.array(v1)
    distances = []
    
    if len(np.shape(array_of_vectors)) == 1:
        array_of_vectors = [array_of_vectors]
        
    
    for v2 in array_of_vectors:
        v2 = np.array(v2)
        if v1.shape != v2.shape:
            raise ValueError(f"Shapes don't match: v1.shape: {v1.shape}, v2.shape: {v2.shape} !")
        
        dist = np.sqrt(np.sum((v1-v2) ** 2))
        
        distances.append(dist)
    return [float(x) for x in distances]

In [5]:
v1 = [1, 2]
v2 = [1, 1]
array_v = [[3, 2], [5, 6]]


In [6]:
cosine_v1_v2 = cosine_similarity(v1, v2)
print(f"Cosine Similarity between v1 and v2: {cosine_v1_v2}")


Cosine Similarity between v1 and v2: [0.9486832980505138]


In [7]:
cosine_v1_array_v = cosine_similarity(v1, array_v)
print(f"Cosine Similarities between v1 and array_v: {cosine_v1_array_v}")


Cosine Similarities between v1 and array_v: [0.8682431421244593, 0.973417168333576]


In [8]:
euclidean_v1_v2 = euclidean_distance(v1, v2)
print(f"Euclidean Distance between v1 and v2: {euclidean_v1_v2}")


Euclidean Distance between v1 and v2: [1.0]


In [9]:
euclidean_v1_array_v = euclidean_distance(v1, array_v)
print(f"Euclidean Distances between v1 and array_v: {euclidean_v1_array_v}")

Euclidean Distances between v1 and array_v: [2.0, 5.656854249492381]


### The embedding model:

The embedding model is responsible of converting a word or a sentence into a fixed size vector. It is trained on millions of samples and is specialized in grouping semantically related sentences (or words).

In this example we use BAA/bge-base-en-v1.5 model.

In [13]:
from sentence_transformers import SentenceTransformer

model_name = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [15]:
# convert a sentence to an embedding vector
res = model.encode("RAG is awesome")

print(res.shape)

(768,)


In [16]:
res

array([ 8.86310544e-03, -4.77513820e-02, -1.56084588e-03,  1.30999703e-02,
       -2.06929748e-03, -6.15726076e-02,  1.38468891e-02,  1.01498771e-03,
       -4.90394719e-02, -4.76255603e-02, -3.62817943e-02,  4.78040241e-03,
       -3.49218771e-02,  5.32315001e-02,  2.19395831e-02,  3.64514701e-02,
        4.02936041e-02, -4.53638146e-03,  1.88379902e-02, -3.36738378e-02,
        2.51618661e-02, -4.84362245e-02, -4.04794067e-02,  2.59089898e-02,
        2.17523947e-02,  3.16036567e-02,  3.93794253e-02, -3.64044793e-02,
       -3.11330333e-02, -1.24722645e-02,  3.66165005e-02, -4.58200322e-03,
       -1.00168621e-03, -3.18878405e-02,  2.95713954e-02,  1.98614895e-02,
       -7.37466477e-03,  2.37017311e-02, -2.15162486e-02, -7.36135095e-02,
       -1.72355585e-02, -1.98693387e-02,  4.12972942e-02,  4.90014767e-03,
       -2.67738625e-02,  2.02293545e-02,  2.96131279e-02,  4.92579490e-02,
       -1.27717601e-02, -3.43212299e-02, -1.39998645e-02,  5.28212823e-02,
       -4.77830414e-03, -

In [18]:
# we can also encode an array of string

array_strings = ['apple', 'car', 'house']

res = model.encode(array_strings)

print(res.shape, "\n", res)

(3, 768) 
 [[-0.02102229  0.06316215  0.00100858 ... -0.00027873  0.00038428
  -0.02818082]
 [ 0.00472379  0.01583519  0.04418226 ... -0.05767498  0.04269494
  -0.00054623]
 [ 0.03586901  0.0641119   0.0230194  ... -0.02925247  0.00125598
   0.02668803]]


<b>In practise: 

In [19]:
words = ['apple', 'car', 'fruit', 'automobile', 'love', 'sentiment']

vectorized_words = model.encode(words)

In [24]:
# On mesure la similarité (cosinsimilarity et euclidean distance) entre le mot "apple" et le reste des mots

word = "apple"
print(f"{word}: ")
for i, w in enumerate(words):
    vectorized_word = vectorized_words[words.index(word)]
    print(f"\t{w}:\t\tCosine Similarity: {cosine_similarity(vectorized_word, vectorized_words[i])[0]:.4f}")

print("\n\n\n")

for i, w in enumerate(words):
    vectorized_word = vectorized_words[words.index(word)]
    print(f"\t{w}:\t\tEuclidean Distance: {euclidean_distance(vectorized_word, vectorized_words[i])[0]:.4f}")


apple: 
	apple:		Cosine Similarity: 1.0000
	car:		Cosine Similarity: 0.5749
	fruit:		Cosine Similarity: 0.7461
	automobile:		Cosine Similarity: 0.6485
	love:		Cosine Similarity: 0.5540
	sentiment:		Cosine Similarity: 0.5020




	apple:		Euclidean Distance: 0.0000
	car:		Euclidean Distance: 0.9221
	fruit:		Euclidean Distance: 0.7126
	automobile:		Euclidean Distance: 0.8384
	love:		Euclidean Distance: 0.9445
	sentiment:		Euclidean Distance: 0.9980


In [25]:
# For documents:
def retrieve_relevant(query, documents, metric='cosine_similarity'):
    query_emb = model.encode(query)
    documents_emb = model.encode(documents)
    vals = []

    if metric == 'cosine_similarity':
        distances = cosine_similarity(query_emb, documents_emb)
        vals = [(doc, dist) for doc, dist in zip(documents, distances)]
        # Sort in descending order
        vals.sort(reverse=True, key=lambda x: x[1])
        
    elif metric == 'euclidean':
        distances = euclidean_distance(query_emb, documents_emb)
        vals = [(doc, dist) for doc, dist in zip(documents, distances)]
        # Sort in ascending order
        vals.sort(key=lambda x: x[1])
        
    return vals

In [26]:
documents = [
    "Mt. Fuji is a breathtaking place to explore during autumn.",
    "Santorini offers stunning views to admire during spring.",
    "Banff National Park is a picturesque destination to visit in the summer.",
    "The Great Wall of China is a spectacular site to experience during winter.",
    "The fjords of Norway are a magical place to cruise through in the spring.",
    "Prague is an enchanting city to wander through in winter.",
    "Kyoto's cherry blossoms create a beautiful scene to witness during spring.",
    "Marrakech offers vibrant markets and culture to enjoy in the fall.",
    "The Maldives are a paradisiacal getaway to savor during summer.",
    "The Christmas markets in Vienna are a festive delight to explore in winter."
]

query = "Suggest to me great places to visit in Asia."


In [27]:
score = retrieve_relevant(query, documents, metric='cosine_similarity')
score

[('The Great Wall of China is a spectacular site to experience during winter.',
  0.6080519556999207),
 ('Mt. Fuji is a breathtaking place to explore during autumn.',
  0.5821490287780762),
 ('The Maldives are a paradisiacal getaway to savor during summer.',
  0.5604647994041443),
 ('Santorini offers stunning views to admire during spring.',
  0.5512217879295349),
 ('Banff National Park is a picturesque destination to visit in the summer.',
  0.5220370888710022),
 ("Kyoto's cherry blossoms create a beautiful scene to witness during spring.",
  0.5150064826011658),
 ('The fjords of Norway are a magical place to cruise through in the spring.',
  0.48470327258110046),
 ('The Christmas markets in Vienna are a festive delight to explore in winter.',
  0.471091628074646),
 ('Marrakech offers vibrant markets and culture to enjoy in the fall.',
  0.4674418866634369),
 ('Prague is an enchanting city to wander through in winter.',
  0.4483725130558014)]

In [28]:
score = retrieve_relevant(query, documents, metric='euclidean')
score

[('The Great Wall of China is a spectacular site to experience during winter.',
  0.8853790760040283),
 ('Mt. Fuji is a breathtaking place to explore during autumn.',
  0.9141673445701599),
 ('The Maldives are a paradisiacal getaway to savor during summer.',
  0.937587559223175),
 ('Santorini offers stunning views to admire during spring.',
  0.9473947286605835),
 ('Banff National Park is a picturesque destination to visit in the summer.',
  0.9777146577835083),
 ("Kyoto's cherry blossoms create a beautiful scene to witness during spring.",
  0.9848793745040894),
 ('The fjords of Norway are a magical place to cruise through in the spring.',
  1.015181541442871),
 ('The Christmas markets in Vienna are a festive delight to explore in winter.',
  1.0285022258758545),
 ('Marrakech offers vibrant markets and culture to enjoy in the fall.',
  1.032044768333435),
 ('Prague is an enchanting city to wander through in winter.',
  1.0503594875335693)]